# V6 — 08: Eval Ablations & CC Variants

**Round 1**: N1, A1, A2, A4  → m=100 (4 GPUs)

**Round 2**: A4cc → m=100, P1cc → m=50 + m=100 (2 GPUs)

| Tag   | Policy              | Round | m    |
|-------|---------------------|-------|------|
| N1    | ACT+ICPE            | 1     | 100  |
| A1    | ACM3+ICPE[sincos]   | 1     | 100  |
| A2    | ACM3+ICPE[linear]   | 1     | 100  |
| A4    | ACM3+SSCP[inf]      | 1     | 100  |
| A4cc  | ACM3+SSCP[CC]       | 2     | 100  |
| P1cc  | ACM3+ICPE+SSCP[CC]  | 2     | 50+100 |

In [ ]:
import sys, os
from pathlib import Path

GPU_OFFSET = 0  # ← only line to change

sys.path.insert(0, str(Path(".").resolve()))
import common_v6 as v6

os.environ.setdefault("MUJOCO_GL", "osmesa")
os.environ.setdefault("PYOPENGL_PLATFORM", "osmesa")

TAGS_R1  = ["N1", "A1", "A2", "A4"]
TAGS_ALL = ["N1", "A1", "A2", "A4", "A4cc", "P1cc"]

# Round 1: 4 models × m=100
EVAL_JOBS_R1 = [(tag, i, 100) for i, tag in enumerate(TAGS_R1)]

# Round 2: A4cc m=100 on GPU+0, P1cc m=50 on GPU+1 (then m=100 on same GPU)
EVAL_JOBS_R2a = [("A4cc", 0, 100), ("P1cc", 1, 50)]
EVAL_JOBS_R2b = [("P1cc", 1, 100)]  # P1cc m=100 after m=50 finishes

In [ ]:
print("=== Training status ===")
v6.print_training_status(TAGS_ALL)
print()
print("=== Eval status ===")
v6.print_eval_status(TAGS_ALL)

In [ ]:
print("Launching Round 1 (N1, A1, A2, A4 @ m=100) ...")
procs_r1 = v6.launch_eval(EVAL_JOBS_R1, GPU_OFFSET)
for tag in TAGS_R1:
    print(f"  tail -f {v6.get_output_dir(tag) / 'eval' / 'eval.log'}")

In [ ]:
# Run Round 2 after Round 1 finishes.
# A4cc m=100 (GPU+0) and P1cc m=50 (GPU+1) run in parallel.
print("Launching Round 2a (A4cc m=100, P1cc m=50) ...")
procs_r2a = v6.launch_eval(EVAL_JOBS_R2a, GPU_OFFSET)
for tag in ["A4cc", "P1cc"]:
    print(f"  tail -f {v6.get_output_dir(tag) / 'eval' / 'eval.log'}")

In [ ]:
# Run after Round 2a finishes.
print("Launching Round 2b (P1cc m=100) ...")
procs_r2b = v6.launch_eval(EVAL_JOBS_R2b, GPU_OFFSET)
print(f"  tail -f {v6.get_output_dir('P1cc') / 'eval' / 'eval.log'}")

In [ ]:
# Post-processing
results = {}
for tag in TAGS_ALL:
    m = v6.build_metrics_from_eval_info(tag)
    results[tag] = m
    v6.save_metrics(tag, m)

print(f"{'TAG':<8} {'LABEL':<35} {'SR':>6} {'CI_LO':>7} {'CI_HI':>7}")
print("-" * 70)
for tag in TAGS_ALL:
    m = results.get(tag, {})
    sr_s = f"{m['sr']:.3f}"      if m.get('sr')       is not None else " ─"
    lo_s = f"{m['sr_ci_lo']:.3f}" if m.get('sr_ci_lo') is not None else " ─"
    hi_s = f"{m['sr_ci_hi']:.3f}" if m.get('sr_ci_hi') is not None else " ─"
    print(f"{tag:<8} {m.get('label', tag):<35} {sr_s:>6} {lo_s:>7} {hi_s:>7}")